## Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns


# Import custom modules
import sys

sys.path.insert(0, "../")


# Set random seed
RANDOM_SEED = 1234
np.random.seed(RANDOM_SEED)

# Configure plotting
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")


## Data Loading Functions

In [ ]:
df = pd.read_csv("../data/diabetic_data.csv")

In [ ]:
df["readmitted"].value_counts() / len(df)
counts = df["readmitted"].value_counts()

percentages = counts * 100 / len(df)

fig, ax = plt.subplots()
percentages.plot(kind="pie", ax=ax, autopct="%1.1f%%")
plt.show()

In [ ]:
# define outcome variable
df = df.replace({"readmitted": {"<30": 1, ">30": 0, "NO": 0}})
print(df["readmitted"].value_counts())

In [ ]:
def age_to_midpoint(age_bucket):
    """
    Convert age buckets like "[70-80)" to numeric midpoint (75).

    Args:
        age_bucket: String representing age range

    Returns:
        Numeric midpoint of the age range
    """
    if pd.isna(age_bucket):
        return np.nan
    try:
        lo, hi = age_bucket.strip("[]").split("-")
        lo = int(lo)
        hi = int(hi.strip(")"))
        return (lo + hi) / 2
    except Exception:
        return np.nan

In [ ]:
# 1. Process age to continuous midpoint
df["age_mid"] = df["age"].apply(age_to_midpoint)
df.drop(columns=["age"], inplace=True)

In [ ]:
# drop columns that are majority missing
df = df.drop(["weight", "payer_code", "medical_specialty"], axis=1)

In [ ]:
# drop missing values in diagnosis columns
df = df.dropna(subset=["diag_1", "diag_2", "diag_3"])
df = df.drop(df.loc[df["diag_1"] == "?"].index, axis=0)
df = df.drop(df.loc[df["diag_2"] == "?"].index, axis=0)
df = df.drop(df.loc[df["diag_3"] == "?"].index, axis=0)

# drop missing values in race column
df = df.dropna(subset=["race"])
df = df.drop(df.loc[df["race"] == "?"].index, axis=0)

# drop missing values in gender column
df = df.drop(df.loc[df["gender"] == "Unknown/Invalid"].index, axis=0)

# drop discharge_disposition_id values that are not relevant for this analysis (expired, hospice, etc.)
df = df.drop(
    df.loc[df["discharge_disposition_id"].isin([11, 19, 20, 21, 25, 26])].index, axis=0
)

# drop drugs with same values for all patients (no predictive power)
df = df.drop(["citoglipton", "examide"], axis=1)


In [ ]:
# Check for missing values in the data
for col in df.columns:
    if df[col].dtype == object:
        print(col, df[col][df[col] == "?"].count())

print("gender", df["gender"][df["gender"] == "Unknown/Invalid"].count())

In [ ]:
df.shape

# Feature Engineering #

In [ ]:
# Create service utilization feature (sum of outpatient, emergency, inpatient visits)
df["service_utilization"] = (
    df["number_outpatient"].fillna(0)
    + df["number_emergency"].fillna(0)
    + df["number_inpatient"].fillna(0)
)

In [ ]:
# Create medication change feature
keys = [
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "glipizide",
    "glyburide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "insulin",
    "glyburide-metformin",
    "tolazamide",
    "metformin-pioglitazone",
    "metformin-rosiglitazone",
    "glimepiride-pioglitazone",
    "glipizide-metformin",
    "troglitazone",
    "tolbutamide",
    "acetohexamide",
]
for col in keys:
    colname = str(col) + "temp"
    df[colname] = df[col].apply(lambda x: 0 if (x == "No" or x == "Steady") else 1)
df["numchange"] = 0
for col in keys:
    colname = str(col) + "temp"
    df["numchange"] = df["numchange"] + df[colname]
    del df[colname]

df["numchange"].value_counts()

In [ ]:
# encode medication changes to binary
df["change"] = df["change"].replace("Ch", 1)
df["change"] = df["change"].replace("No", 0)
df["gender"] = df["gender"].replace("Male", 1)
df["gender"] = df["gender"].replace("Female", 0)
df["diabetesMed"] = df["diabetesMed"].replace("Yes", 1)
df["diabetesMed"] = df["diabetesMed"].replace("No", 0)
# keys is the same as before
for col in keys:
    df[col] = df[col].replace("No", 0)
    df[col] = df[col].replace("Steady", 1)
    df[col] = df[col].replace("Up", 1)
    df[col] = df[col].replace("Down", 1)

In [ ]:
# encode A1Cresult and max_glu_serum
df["A1Cresult"] = df["A1Cresult"].replace(">7", 1)
df["A1Cresult"] = df["A1Cresult"].replace(">8", 1)
df["A1Cresult"] = df["A1Cresult"].replace("Norm", 0)
df["A1Cresult"] = df["A1Cresult"].replace("None", -99)
df["A1Cresult"] = df["A1Cresult"].fillna(-99)  # Fill NaN values with -99

df["max_glu_serum"] = df["max_glu_serum"].replace(">200", 1)
df["max_glu_serum"] = df["max_glu_serum"].replace(">300", 1)
df["max_glu_serum"] = df["max_glu_serum"].replace("Norm", 0)
df["max_glu_serum"] = df["max_glu_serum"].replace("None", -99)
df["max_glu_serum"] = df["max_glu_serum"].fillna(-99)  # Fill NaN values with -99

## Diagnosis Processing Functions

In [ ]:
def diagnosis_to_category(diag_code):
    """
    Map diagnosis codes to broader categories.

    Args:
        diag_code: Diagnosis code as string or number

    Returns:
        Diagnosis category as int (1-9, or 0 for Unknown)
    """
    if pd.isna(diag_code) or diag_code == "?" or diag_code is None:
        return 0  # Unknown

    try:
        # Handle string codes that start with V or E
        if isinstance(diag_code, str):
            if diag_code.startswith("V") or diag_code.startswith("E"):
                return 9  # Supplementary
            # Try to convert to float for numeric comparison
            code = float(diag_code)
        else:
            code = float(diag_code)

        # Map to categories based on ICD-9 code ranges
        if 390 <= code <= 459 or code == 785:
            return 1  # Circulatory
        elif 460 <= code <= 519 or code == 786:
            return 2  # Respiratory
        elif 520 <= code <= 579 or code == 787:
            return 3  # Digestive
        elif 250 <= code < 251:
            return 4  # Diabetes
        elif 800 <= code <= 999:
            return 5  # Injury
        elif 710 <= code <= 739:
            return 6  # Musculoskeletal
        elif 580 <= code <= 629 or code == 788:
            return 7  # Genitourinary
        elif 140 <= code <= 239:
            return 8  # Neoplasms
        else:
            return 9  # Other

    except (ValueError, TypeError):
        return 9  # Other for any conversion errors


# Create diagnosis categories
# This mirrors the diagnosis categorization from the original study
df["diag_1_cat"] = df["diag_1"].apply(diagnosis_to_category)
df["diag_2_cat"] = df["diag_2"].apply(diagnosis_to_category)
df["diag_3_cat"] = df["diag_3"].apply(diagnosis_to_category)

print("Diagnosis categories created:")
print(f"diag_1_cat distribution:\n{df['diag_1_cat'].value_counts().sort_index()}")
print(f"\ndiag_2_cat distribution:\n{df['diag_2_cat'].value_counts().sort_index()}")
print(f"\ndiag_3_cat distribution:\n{df['diag_3_cat'].value_counts().sort_index()}")

# Category mapping for reference:
# 0: Unknown/Missing
# 1: Circulatory (390-459, 785)
# 2: Respiratory (460-519, 786)
# 3: Digestive (520-579, 787)
# 4: Diabetes (250-250.99)
# 5: Injury (800-999)
# 6: Musculoskeletal (710-739)
# 7: Genitourinary (580-629, 788)
# 8: Neoplasms (140-239)
# 9: Other/Supplementary

In [ ]:
# Create binary columns for each diagnosis category (0-9)
# This creates indicator variables for whether each patient has a diagnosis in each category

# Initialize all diagnosis category columns to 0
for i in range(10):  # Categories 0-9
    df[f"has_diag_cat_{i}"] = 0

# Set to 1 if patient has any diagnosis in that category
for idx, row in df.iterrows():
    # Check primary diagnosis
    if row["diag_1_cat"] is not None:
        df.loc[idx, f"has_diag_cat_{int(row['diag_1_cat'])}"] = 1

    # Check secondary diagnosis
    if row["diag_2_cat"] is not None:
        df.loc[idx, f"has_diag_cat_{int(row['diag_2_cat'])}"] = 1

    # Check tertiary diagnosis
    if row["diag_3_cat"] is not None:
        df.loc[idx, f"has_diag_cat_{int(row['diag_3_cat'])}"] = 1

# Show summary of diagnosis category prevalence
print("\nDiagnosis category prevalence (binary indicators):")
diag_cols = [f"has_diag_cat_{i}" for i in range(10)]
for col in diag_cols:
    count = df[col].sum()
    pct = (count / len(df)) * 100
    print(f"{col}: {count} patients ({pct:.1f}%)")

# Category mapping for reference:
category_names = {
    0: "Unknown/Missing",
    1: "Circulatory",
    2: "Respiratory",
    3: "Digestive",
    4: "Diabetes",
    5: "Injury",
    6: "Musculoskeletal",
    7: "Genitourinary",
    8: "Neoplasms",
    9: "Other/Supplementary",
}

print(f"\nTotal patients: {len(df)}")
print(f"Shape after adding diagnosis categories: {df.shape}")

## Categorical Data Grouping Functions

In [ ]:
# Group discharge disposition into broader categories
# we already deleted expired/hospice/etc. values above [11, 19, 20, 21, 25, 26]


def group_discharge_disposition(df):
    # 1 = Home
    # 2 = SNF-like
    # 10 = Other
    mapping = {
        1: 1,  # Home
        6: 1,  # Home (hospice)
        8: 1,  # Home (left AMA - treat as home)
        9: 1,
        13: 1,  # Skilled nursing facility
        3: 2,  # Intermediate care facility (SNF-like)
        4: 2,  # Another type of facility (SNF-like)
        5: 2,
        14: 2,
        22: 2,
        23: 2,
        24: 2,
        12: 10,
        15: 10,
        16: 10,
        17: 10,
        # Medical facility (SNF-like)
        # Others (expired, etc.) will be marked as missing/other
    }
    df["discharge_disposition_grouped"] = df["discharge_disposition_id"].map(mapping)
    df["discharge_disposition_grouped"] = df["discharge_disposition_grouped"].fillna(
        df["discharge_disposition_id"]
    )  # fill missing with original value
    df.drop(columns=["discharge_disposition_id"], inplace=True)

    return df

In [ ]:
def group_admission_type_id(df):
    """
    Group admission types into 3 categories to match study.
    Categories: 1=Emergency, 2=Urgent, 3=Elective

    Args:
        df: DataFrame with 'admission_type_id' column

    Returns:
        DataFrame with 'admission_type_grouped' column
    """

    if "admission_type_id" in df.columns:
        # Study uses: Emergency, Urgent, Elective
        mapping = {  # Emergency
            2: 1,  # Urgent
            6: 5,  # NULL (default to urgent)
            7: 1,  # Trauma Center (emergency)
            8: 5,  # Not Mapped (default to emergency)
        }
        df["admission_type_grouped"] = df["admission_type_id"].map(mapping)
        df["admission_type_grouped"] = df["admission_type_grouped"].fillna(
            df["admission_type_id"]
        )  # fill missing with original value
        df.drop(columns=["admission_type_id"], inplace=True)

    return df

In [ ]:
def grouped_admission_source_id(df):
    """
    Group admission source ids into broader categories.

    Args:
        df: DataFrame with 'admission_source_id' column

    Returns:
        DataFrame with 'admission_source_grouped' column
    """

    if "admission_source_id" in df.columns:
        mapping = {  # Emergency
            2: 1,
            3: 1,
            5: 4,
            6: 4,
            10: 4,
            22: 4,
            25: 4,
            15: 9,
            17: 9,
            20: 9,
            21: 9,
            13: 11,
            14: 11,
        }
        df["admission_source_grouped"] = df["admission_source_id"].map(mapping)
        df["admission_source_grouped"] = df["admission_source_grouped"].fillna(
            df["admission_source_id"]
        )  # fill missing with original value
        df.drop(columns=["admission_source_id"], inplace=True)
    return df

## Data Cleaning and Encoding

In [ ]:
df_cleaned = df.drop_duplicates(
    subset=["patient_nbr"], keep="first"
)  # keep first admission only
df_cleaned = df_cleaned.drop(
    ["encounter_id", "patient_nbr"], axis=1
)  # drop identifiers
print(f"After removing duplicates and identifiers: {df_cleaned.shape}")
df_cleaned.head()

In [ ]:
# Apply categorical grouping functions
df_encoded = df_cleaned.copy()

# Group discharge dispositions
df_encoded = group_discharge_disposition(df_encoded)

# Group admission types
df_encoded = group_admission_type_id(df_encoded)

# Group admission sources
df_encoded = grouped_admission_source_id(df_encoded)

print(f"After grouping categorical features: {df_encoded.shape}")
print("\nGrouped features:")
print(
    f"discharge_disposition_grouped unique values: {df_encoded['discharge_disposition_grouped'].nunique()}"
)
print(
    f"admission_type_grouped unique values: {df_encoded['admission_type_grouped'].nunique()}"
)
print(
    f"admission_source_grouped unique values: {df_encoded['admission_source_grouped'].nunique()}"
)

In [ ]:
# One-hot encode the grouped categorical variables
categorical_columns = [
    "discharge_disposition_grouped",
    "admission_type_grouped",
    "admission_source_grouped",
    "race",
]

print("Before one-hot encoding:", df_encoded.shape)
df_encoded = pd.get_dummies(
    df_encoded, columns=categorical_columns, prefix=categorical_columns, dtype=int
)
print("After one-hot encoding:", df_encoded.shape)

# Show the new column names created
new_cols = [
    col for col in df_encoded.columns if any(cat in col for cat in categorical_columns)
]
print(f"\nCreated {len(new_cols)} one-hot encoded columns")
print("Sample columns:", new_cols[:10])

In [ ]:
# Drop the original diagnosis columns (we have diagnosis categories now)
df_encoded = df_encoded.drop(["diag_1", "diag_2", "diag_3"], axis=1)

# Drop the categorical diagnosis columns (we have binary indicators now)
df_encoded = df_encoded.drop(["diag_1_cat", "diag_2_cat", "diag_3_cat"], axis=1)

print(f"After dropping original diagnosis columns: {df_encoded.shape}")

In [ ]:
df_encoded = df_encoded.drop(
    "has_diag_cat_0", axis=1
)  # no examples with unknown diagnosis
df_encoded.shape

### Feature Summary

Features after encoding:

In [ ]:
# Comprehensive feature summary
print("=" * 70)
print("FEATURE ENCODING SUMMARY")
print("=" * 70)

print(f"\nTotal features after encoding: {df_encoded.shape[1]}")
print(f"Total samples: {df_encoded.shape[0]}")

# Categorize features by type
print("\n--- FEATURE CATEGORIES ---\n")

# Continuous/Numeric features
continuous_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "age_mid",
    "service_utilization",
    "numchange",
]
continuous_in_df = [f for f in continuous_features if f in df_encoded.columns]
print(f"Continuous features ({len(continuous_in_df)}):")
for f in continuous_in_df:
    print(f"  - {f}")

# Binary features (already encoded)
binary_features = ["gender", "change", "diabetesMed", "A1Cresult", "max_glu_serum"]
medication_features = [col for col in df_encoded.columns if col in keys]
binary_in_df = [
    f for f in binary_features if f in df_encoded.columns
] + medication_features
print(f"\nBinary features ({len(binary_in_df)}):")
for f in binary_in_df[:15]:  # Show first 15
    print(f"  - {f}")
if len(binary_in_df) > 15:
    print(f"  ... and {len(binary_in_df) - 15} more medication features")

# Diagnosis category binary indicators
diag_features = [col for col in df_encoded.columns if col.startswith("has_diag_cat_")]
print(f"\nDiagnosis category indicators ({len(diag_features)}):")
for f in diag_features:
    cat_num = f.split("_")[-1]
    count = df_encoded[f].sum()
    pct = (count / len(df_encoded)) * 100
    print(f"  - {f}: {count} patients ({pct:.1f}%)")

# One-hot encoded features
onehot_features = [
    col
    for col in df_encoded.columns
    if any(
        x in col
        for x in [
            "discharge_disposition_grouped_",
            "admission_type_grouped_",
            "admission_source_grouped_",
            "race_",
        ]
    )
]
print(f"\nOne-hot encoded features ({len(onehot_features)}):")
for prefix in [
    "discharge_disposition_grouped",
    "admission_type_grouped",
    "admission_source_grouped",
    "race",
]:
    prefix_cols = [col for col in onehot_features if col.startswith(prefix + "_")]
    if prefix_cols:
        print(f"  {prefix}: {len(prefix_cols)} categories")

# Target variable
print(f"\nTarget variable: readmitted")
print(
    f"  Class 0 (No readmission <30 days): {(df_encoded['readmitted'] == 0).sum()} ({(df_encoded['readmitted'] == 0).sum() / len(df_encoded) * 100:.1f}%)"
)
print(
    f"  Class 1 (Readmission <30 days): {(df_encoded['readmitted'] == 1).sum()} ({(df_encoded['readmitted'] == 1).sum() / len(df_encoded) * 100:.1f}%)"
)

print("\n" + "=" * 70)
print(
    f"READY FOR MODELING: {df_encoded.shape[1] - 1} features, {df_encoded.shape[0]} samples"
)
print("=" * 70)

In [ ]:
df_encoded.head()


### Prepare for Train/Test Split

Now we can split the data and standardize continuous features:

In [ ]:
# Separate features and target
X = df_encoded.drop("readmitted", axis=1)
y = df_encoded["readmitted"]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns ({len(X.columns)}):")
print(X.columns.tolist()[:20], "... and more")

In [ ]:
# Shuffle the data first
from sklearn.utils import shuffle as sklearn_shuffle

X_shuffled = sklearn_shuffle(X, random_state=RANDOM_SEED)
y_shuffled = sklearn_shuffle(y, random_state=RANDOM_SEED)

# Ensure X and y are aligned after shuffling
X_shuffled = X_shuffled.reset_index(drop=True)
y_shuffled = y_shuffled.reset_index(drop=True)

print(f"Data shuffled with random_state={RANDOM_SEED}")
print(f"Features shape: {X_shuffled.shape}")
print(f"Target shape: {y_shuffled.shape}")

In [ ]:
# Split data into train (60%), validation (20%), and test (20%) sets
# First split: 60% train, 40% temp (which will become val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_shuffled, y_shuffled, test_size=0.4, random_state=RANDOM_SEED, stratify=y_shuffled
)

# Second split: split the 40% into 20% val and 20% test (50/50 of the temp set)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp
)

print(f"Training set: {X_train.shape} ({len(X_train) / len(X) * 100:.1f}%)")
print(f"Validation set: {X_val.shape} ({len(X_val) / len(X) * 100:.1f}%)")
print(f"Test set: {X_test.shape} ({len(X_test) / len(X) * 100:.1f}%)")

print("Training target distribution:")
print(y_train.value_counts())
print(f"  Class 0: {(y_train == 0).sum() / len(y_train) * 100:.1f}%")
print(f"  Class 1: {(y_train == 1).sum() / len(y_train) * 100:.1f}%")

print("Validation target distribution:")
print(y_val.value_counts())
print(f"  Class 0: {(y_val == 0).sum() / len(y_val) * 100:.1f}%")
print(f"  Class 1: {(y_val == 1).sum() / len(y_val) * 100:.1f}%")

print("Test target distribution:")
print(y_test.value_counts())
print(f"  Class 0: {(y_test == 0).sum() / len(y_test) * 100:.1f}%")
print(f"  Class 1: {(y_test == 1).sum() / len(y_test) * 100:.1f}%")

In [ ]:
# Standardize continuous features using MinMaxScaler
continuous_features_to_scale = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "age_mid",
    "service_utilization",
    "numchange",
]

# Only scale features that exist in the dataset
continuous_features_to_scale = [
    f for f in continuous_features_to_scale if f in X_train.columns
]

scaler = MinMaxScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

# Fit on training data only, then transform all three sets
X_train_scaled[continuous_features_to_scale] = scaler.fit_transform(
    X_train[continuous_features_to_scale]
)
X_val_scaled[continuous_features_to_scale] = scaler.transform(
    X_val[continuous_features_to_scale]
)
X_test_scaled[continuous_features_to_scale] = scaler.transform(
    X_test[continuous_features_to_scale]
)

print(f"Scaled {len(continuous_features_to_scale)} continuous features:")
for feat in continuous_features_to_scale:
    print(
        f"  - {feat}: train range [{X_train_scaled[feat].min():.3f}, {X_train_scaled[feat].max():.3f}]"
    )

print(f"\n✓ Training data ready: {X_train_scaled.shape}")
print(f"✓ Validation data ready: {X_val_scaled.shape}")
print(f"✓ Test data ready: {X_test_scaled.shape}")
print(f"\n{'=' * 70}")
print("DATA PREPARATION COMPLETE - READY FOR MODELING")
print(f"{'=' * 70}")

In [ ]:
# Save the processed datasets for modeling
X_train_scaled.to_csv("../data/X_train_scaled.csv", index=False)
X_val_scaled.to_csv("../data/X_val_scaled.csv", index=False)
X_test_scaled.to_csv("../data/X_test_scaled.csv", index=False)

y_train.to_csv("../data/y_train.csv", index=False)
y_val.to_csv("../data/y_val.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)

print("✓ Saved training data:")
print(f"  - X_train_scaled.csv: {X_train_scaled.shape}")
print(f"  - y_train.csv: {y_train.shape}")

print("\n✓ Saved validation data:")
print(f"  - X_val_scaled.csv: {X_val_scaled.shape}")
print(f"  - y_val.csv: {y_val.shape}")

print("\n✓ Saved test data:")
print(f"  - X_test_scaled.csv: {X_test_scaled.shape}")
print(f"  - y_test.csv: {y_test.shape}")

print("\n" + "=" * 70)
print("All datasets saved to '../data/' directory")
print("=" * 70)